# Batch Data Processing from a public API



In [1]:
import os
from typing import Dict, Any, Callable

from dotenv import load_dotenv
import json
import requests 

In [2]:
load_dotenv('./src/env', override=True)

CLIENT_ID = os.getenv('CLIENT_ID')
CLIENT_SECRET = os.getenv('CLIENT_SECRET')

In [3]:
def get_token(client_id: str, client_secret: str, url: str) -> Dict[Any, Any]:
    """Allows to perform a POST request to obtain an access token 

    Args:
        client_id (str): App client id
        client_secret (str): App client secret
        url (str): URL to perform the post request

    Returns:
        Dict[Any, Any]: Dictionary containing the access token
    """
        
    headers = {        
        "Content-Type": "application/x-www-form-urlencoded"            
    }
    
    payload = {
                "grant_type": "client_credentials", 
                "client_id": client_id, 
                "client_secret": client_secret
               }
    
    try: 
        response = requests.post(url=url, headers=headers, data=payload)
        print(type(response))
        response.raise_for_status()
        response_json = json.loads(response.content)
        
        return response_json
        
    except Exception as err:
        print(f"Error: {err}")
        return {}

URL_TOKEN="https://accounts.spotify.com/api/token"
token = get_token(client_id=CLIENT_ID, client_secret=CLIENT_SECRET, url=URL_TOKEN)

print(token)

<class 'requests.models.Response'>
{'access_token': 'BQAgyBp_rekJGhe4SDc8pflWkIANguKRlpcF-IZkh3UFNyo1Ne9h9yV_fpp27cp779Y5NGGWc2iE9lk-Qj4mjqEYFcVBOWsNZ_KJEgb0wysxrXHnRRZhhqIrg5r-hh1b6fYRTwgPyhs', 'token_type': 'Bearer', 'expires_in': 3600}


In [4]:
def get_auth_header(access_token: str) -> Dict[str, str]:
    return {"Authorization": f"Bearer {access_token}"}

In [8]:
response = requests.get(url = "https://api.spotify.com/v1/browse/new-releases" , headers = get_auth_header(token.get('access_token')))

In [9]:
response = response.json()

In [10]:
response.keys()

dict_keys(['albums'])

In [11]:
response.get('albums').keys()

dict_keys(['href', 'items', 'limit', 'next', 'offset', 'previous', 'total'])

In [12]:
response.get('albums').get('href')

'https://api.spotify.com/v1/browse/new-releases?offset=0&limit=20'

In [13]:
response.get('albums').get('items')

[{'album_type': 'album',
  'artists': [{'external_urls': {'spotify': 'https://open.spotify.com/artist/06HL4z0CvFAxyc27GXpf02'},
    'href': 'https://api.spotify.com/v1/artists/06HL4z0CvFAxyc27GXpf02',
    'id': '06HL4z0CvFAxyc27GXpf02',
    'name': 'Taylor Swift',
    'type': 'artist',
    'uri': 'spotify:artist:06HL4z0CvFAxyc27GXpf02'}],
  'available_markets': ['AR',
   'AU',
   'AT',
   'BE',
   'BO',
   'BR',
   'BG',
   'CA',
   'CL',
   'CO',
   'CR',
   'CY',
   'CZ',
   'DK',
   'DO',
   'DE',
   'EC',
   'EE',
   'SV',
   'FI',
   'FR',
   'GR',
   'GT',
   'HN',
   'HK',
   'HU',
   'IS',
   'IE',
   'IT',
   'LV',
   'LT',
   'LU',
   'MY',
   'MT',
   'MX',
   'NL',
   'NZ',
   'NI',
   'NO',
   'PA',
   'PY',
   'PE',
   'PH',
   'PL',
   'PT',
   'SG',
   'SK',
   'ES',
   'SE',
   'CH',
   'TW',
   'TR',
   'UY',
   'US',
   'GB',
   'AD',
   'LI',
   'MC',
   'ID',
   'JP',
   'TH',
   'VN',
   'RO',
   'IL',
   'ZA',
   'SA',
   'AE',
   'BH',
   'QA',
   'OM',
   'KW',

In [14]:
print(response.get('albums').get('offset'))

0


In [15]:
print(response.get('albums').get('limit'))

20


In [16]:
print(response.get('albums').get('total'))

100


In [17]:
response.get('albums').get('next')

'https://api.spotify.com/v1/browse/new-releases?offset=20&limit=20'

In [18]:
response = requests.get(url = "https://api.spotify.com/v1/browse/new-releases?offset=20&limit=20" , headers = get_auth_header(token.get('access_token')))

In [21]:
def get_new_releases(url: str, access_token: str, offset: int = 0, limit: int = 20, next: str = "") -> Dict[Any, Any]:
    if next == "":
        request_url = f"{url}?offset={offset}&limit={limit}"
    else:
        request_url = f"{next}"
    headers = get_auth_header(access_token)

    try:
        response = requests.get(url=request_url, headers=headers)
        return response.json()
    except Exception as err:
        print(f"Error requesting data: {err}")
        return {'error': err}

In [26]:
URL_NEW_RELEASES = "https://api.spotify.com/v1/browse/new-releases"

releases_response = get_new_releases(
    url=URL_NEW_RELEASES,
    access_token=token.get('access_token')
)


In [27]:
releases_response.keys()

dict_keys(['albums'])

In [28]:
releases_response.get('albums').keys()

dict_keys(['href', 'items', 'limit', 'next', 'offset', 'previous', 'total'])

In [29]:
releases_response.get('albums').get('total')

100

In [30]:
len(releases_response.get('albums').get('items'))

20

Explore the items:

In [31]:
releases_response.get('albums').get('items')[0]

{'album_type': 'album',
 'artists': [{'external_urls': {'spotify': 'https://open.spotify.com/artist/06HL4z0CvFAxyc27GXpf02'},
   'href': 'https://api.spotify.com/v1/artists/06HL4z0CvFAxyc27GXpf02',
   'id': '06HL4z0CvFAxyc27GXpf02',
   'name': 'Taylor Swift',
   'type': 'artist',
   'uri': 'spotify:artist:06HL4z0CvFAxyc27GXpf02'}],
 'available_markets': ['AR',
  'AU',
  'AT',
  'BE',
  'BO',
  'BR',
  'BG',
  'CA',
  'CL',
  'CO',
  'CR',
  'CY',
  'CZ',
  'DK',
  'DO',
  'DE',
  'EC',
  'EE',
  'SV',
  'FI',
  'FR',
  'GR',
  'GT',
  'HN',
  'HK',
  'HU',
  'IS',
  'IE',
  'IT',
  'LV',
  'LT',
  'LU',
  'MY',
  'MT',
  'MX',
  'NL',
  'NZ',
  'NI',
  'NO',
  'PA',
  'PY',
  'PE',
  'PH',
  'PL',
  'PT',
  'SG',
  'SK',
  'ES',
  'SE',
  'CH',
  'TW',
  'TR',
  'UY',
  'US',
  'GB',
  'AD',
  'LI',
  'MC',
  'ID',
  'JP',
  'TH',
  'VN',
  'RO',
  'IL',
  'ZA',
  'SA',
  'AE',
  'BH',
  'QA',
  'OM',
  'KW',
  'EG',
  'MA',
  'DZ',
  'TN',
  'LB',
  'JO',
  'PS',
  'IN',
  'KZ',
  'MD

In [32]:
next_releases_response = get_new_releases(url=URL_NEW_RELEASES, access_token=token.get('access_token'), offset=20, limit=20)

In [33]:
next_releases_response.get('albums').get('href')

'https://api.spotify.com/v1/browse/new-releases?offset=20&limit=20'

In [34]:
next_releases_response.get('albums').get('next')

'https://api.spotify.com/v1/browse/new-releases?offset=40&limit=20'

In [35]:
last_releases_response = get_new_releases(url=URL_NEW_RELEASES, access_token=token.get('access_token'), offset=80, limit=20)

In [36]:
print(last_releases_response.get('albums').get('previous'))
print(last_releases_response.get('albums').get('next'))

https://api.spotify.com/v1/browse/new-releases?offset=60&limit=20
None


In [38]:
def paginated_new_releases(endpoint_request: Callable, url: str, access_token: str, offset: int = 0, limit: int = 20) -> list:
    responses = []

    kwargs = {
        "url": url,
        "access_token": access_token,
        "offset": offset,
        "limit": limit,
    }
    response = endpoint_request(**kwargs)
    responses.extend(response.get('albums', {}).get('items', []))
    total_elements = response.get('albums', {}).get('total', 0)
    while offset + limit < total_elements:
        offset = offset + limit  
        kwargs = {
            "url": url,
            "access_token": access_token,
            "offset": offset,
            "limit": limit,
        }
        response = endpoint_request(**kwargs)
        responses.extend(response.get('albums', {}).get('items', []))
        print(f"Finished iteration for page with offset: {offset - limit}")
    return responses

In [39]:
responses = paginated_new_releases(endpoint_request=get_new_releases,
                                   url=URL_NEW_RELEASES, 
                                   access_token=token.get('access_token'), 
                                   offset=0, limit=20)

Finished iteration for page with offset: 0
Finished iteration for page with offset: 20
Finished iteration for page with offset: 40
Finished iteration for page with offset: 60


Have a look at one of the item:

In [40]:
responses[0]

{'album_type': 'album',
 'artists': [{'external_urls': {'spotify': 'https://open.spotify.com/artist/06HL4z0CvFAxyc27GXpf02'},
   'href': 'https://api.spotify.com/v1/artists/06HL4z0CvFAxyc27GXpf02',
   'id': '06HL4z0CvFAxyc27GXpf02',
   'name': 'Taylor Swift',
   'type': 'artist',
   'uri': 'spotify:artist:06HL4z0CvFAxyc27GXpf02'}],
 'available_markets': ['AR',
  'AU',
  'AT',
  'BE',
  'BO',
  'BR',
  'BG',
  'CA',
  'CL',
  'CO',
  'CR',
  'CY',
  'CZ',
  'DK',
  'DO',
  'DE',
  'EC',
  'EE',
  'SV',
  'FI',
  'FR',
  'GR',
  'GT',
  'HN',
  'HK',
  'HU',
  'IS',
  'IE',
  'IT',
  'LV',
  'LT',
  'LU',
  'MY',
  'MT',
  'MX',
  'NL',
  'NZ',
  'NI',
  'NO',
  'PA',
  'PY',
  'PE',
  'PH',
  'PL',
  'PT',
  'SG',
  'SK',
  'ES',
  'SE',
  'CH',
  'TW',
  'TR',
  'UY',
  'US',
  'GB',
  'AD',
  'LI',
  'MC',
  'ID',
  'JP',
  'TH',
  'VN',
  'RO',
  'IL',
  'ZA',
  'SA',
  'AE',
  'BH',
  'QA',
  'OM',
  'KW',
  'EG',
  'MA',
  'DZ',
  'TN',
  'LB',
  'JO',
  'PS',
  'IN',
  'KZ',
  'MD

In [41]:
len(responses)

100

In [45]:
from typing import Callable

def paginated_with_next_new_releases(endpoint_request: Callable, url: str, access_token: str) -> list:
    """Manages pagination for API requests done with the endpoint_request callable.

    Args:
        endpoint_request (Callable): Function that performs API request
        url (str): Base URL for the request
        access_token (str): Access token

    Returns:
        list: All album items from paginated responses
    """
    responses = []
    next_page = url 

    kwargs = {
        "url": url,
        "access_token": access_token,
        "next": ""  
    }

    while next_page:
        response = endpoint_request(**kwargs)
        responses.extend(response.get('albums', {}).get('items', []))
        next_page = response.get('albums', {}).get('next')
        kwargs["next"] = next_page

        print(f"Executed request with URL: {response.get('albums', {}).get('href')}")

    return responses


In [46]:
responses_with_next = paginated_with_next_new_releases(endpoint_request=get_new_releases, 
                                                             url=URL_NEW_RELEASES, 
                                                             access_token=token.get('access_token'))

Executed request with URL: https://api.spotify.com/v1/browse/new-releases?offset=0&limit=20
Executed request with URL: https://api.spotify.com/v1/browse/new-releases?offset=20&limit=20
Executed request with URL: https://api.spotify.com/v1/browse/new-releases?offset=40&limit=20
Executed request with URL: https://api.spotify.com/v1/browse/new-releases?offset=60&limit=20
Executed request with URL: https://api.spotify.com/v1/browse/new-releases?offset=80&limit=20


In [47]:
responses_with_next[0]

{'album_type': 'album',
 'artists': [{'external_urls': {'spotify': 'https://open.spotify.com/artist/06HL4z0CvFAxyc27GXpf02'},
   'href': 'https://api.spotify.com/v1/artists/06HL4z0CvFAxyc27GXpf02',
   'id': '06HL4z0CvFAxyc27GXpf02',
   'name': 'Taylor Swift',
   'type': 'artist',
   'uri': 'spotify:artist:06HL4z0CvFAxyc27GXpf02'}],
 'available_markets': ['AR',
  'AU',
  'AT',
  'BE',
  'BO',
  'BR',
  'BG',
  'CA',
  'CL',
  'CO',
  'CR',
  'CY',
  'CZ',
  'DK',
  'DO',
  'DE',
  'EC',
  'EE',
  'SV',
  'FI',
  'FR',
  'GR',
  'GT',
  'HN',
  'HK',
  'HU',
  'IS',
  'IE',
  'IT',
  'LV',
  'LT',
  'LU',
  'MY',
  'MT',
  'MX',
  'NL',
  'NZ',
  'NI',
  'NO',
  'PA',
  'PY',
  'PE',
  'PH',
  'PL',
  'PT',
  'SG',
  'SK',
  'ES',
  'SE',
  'CH',
  'TW',
  'TR',
  'UY',
  'US',
  'GB',
  'AD',
  'LI',
  'MC',
  'ID',
  'JP',
  'TH',
  'VN',
  'RO',
  'IL',
  'ZA',
  'SA',
  'AE',
  'BH',
  'QA',
  'OM',
  'KW',
  'EG',
  'MA',
  'DZ',
  'TN',
  'LB',
  'JO',
  'PS',
  'IN',
  'KZ',
  'MD

In [ ]:
def paginated_new_releases_sdk(limit: int = 20) -> list:
    album_data = []
    first_page = spotify.new_releases(limit=limit, offset=0)
    albums_obj = first_page["albums"]
    total_albums_elements = albums_obj["total"]
    offset_idx = list(range(0, total_albums_elements, limit))
    for idx in offset_idx:
        response_page = spotify.new_releases(limit=limit, offset=idx)
        album_data.extend(response_page["albums"]["items"])
    return album_data


In [ ]:
len(album_data_sdk)